In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(dataset)

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)


[{'task': 'Write a Python function that validates an AWS IAM role ARN format and returns True if valid, False otherwise. The ARN should follow the pattern: arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME'}, {'task': "Create a JSON object that represents an AWS S3 bucket policy allowing public read access to all objects in the bucket named 'my-public-bucket'"}, {'task': 'Write a regular expression pattern that matches valid AWS EC2 security group IDs (format: sg- followed by 8 or 17 hexadecimal characters)'}]


In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [9]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [10]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [11]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS IAM Role ARN Validator\n\nHere's a Python function that validates AWS IAM role ARN format:\n\n```python\nimport re\n\ndef validate_iam_role_arn(arn: str) -> bool:\n    \"\"\"\n    Validates an AWS IAM role ARN format.\n    \n    ARN format: arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME\n    \n    Args:\n        arn: The ARN string to validate\n        \n    Returns:\n        True if the ARN is valid, False otherwise\n        \n    Examples:\n        >>> validate_iam_role_arn(\"arn:aws:iam::123456789012:role/MyRole\")\n        True\n        >>> validate_iam_role_arn(\"arn:aws:iam::123456789012:role/path/MyRole\")\n        True\n        >>> validate_iam_role_arn(\"arn:aws:iam::123456789012:user/MyUser\")\n        False\n        >>> validate_iam_role_arn(\"invalid-arn\")\n        False\n    \"\"\"\n    if not isinstance(arn, str):\n        return False\n    \n    # AWS IAM Role ARN pattern\n    # arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME\n    # ACCOUNT_ID: 12 digits\n    